In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic"
os.environ['MKL_THREADING_LAYER'] = "GNU"

In [4]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [6]:
# torch.cuda.set_per_process_memory_fraction(0.5)
# torch.set_num_threads(1)
resource.setrlimit(resource.RLIMIT_AS, (30 * 1024 * 1024 * 1024, -1))


In [7]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [8]:
if is_main:
    if is_jupyter: 
        # Basics 
        seed        = 42
        environment_string = "mini_grid"
        gold_timesteps = 1_000_000
        training_timesteps = 1_000_000
        num_concepts_selected = 24
        selection_function = "q_value"
        # Experiment #1 & #2
        run_basic = False
        run_iterative = False 
        run_two_stage = False  
        run_imperfect=True
        run_intervention=True
        # Experiment #3
        cbm_accuracy_by_concept = None 
        intervention_probability = 0
        intervention_accuracy_by_concept = None 
        cbm_std_by_concept = None 
        target_abstraction = 0.05
        reward_error = 0
        # Experiment #4
        concept_source = "human_selected_binary"
        # Experiment #5
        assess_completeness=False
        # Experiment #6
        num_iterations = 0
        selections_per_round = 0
        initial_concepts = 0
        out_folder = "llm"
    else:
        parser = argparse.ArgumentParser()
        parser.add_argument('--seed', help='Random Seed', type=int, default=42)
        parser.add_argument('--environment_string', help='Which environment to create', type=str, default="tree")
        parser.add_argument('--training_timesteps', help='Number of training timesteps', type=int, default=10000)
        parser.add_argument('--gold_timesteps', help='Number of training timesteps without concepts', type=int, default=10000)
        parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
        parser.add_argument('--selection_function', help='When selecting, use q_value, policy, or transition?', type=str, default="policy")
        parser.add_argument('--cbm_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--intervention_accuracy_by_concept', help="What is the accuracy of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--cbm_std_by_concept', help="What is the error of AI per concept?", nargs='*', type=float, default=None)
        parser.add_argument('--run_two_stage', help='Run the two stage?', action='store_true')
        parser.add_argument('--run_iterative', help='Run the iterative?', action='store_true')
        parser.add_argument('--run_intervention', help='Run the intervention?', action='store_true')
        parser.add_argument('--run_basic', help='Run the basic comparisons?', action='store_true')
        parser.add_argument('--run_imperfect', help='Run the imperfect comparisons?', action='store_true')
        parser.add_argument('--intervention_probability', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--target_abstraction', help='Value for the target abstraction with human performance', type=float, default=0.05)
        parser.add_argument('--reward_error', help="How much to perturb the reward by?", type=float, default=0)
        parser.add_argument('--concept_source', help='When selecting, use q_value, policy, or transition?', type=str, default="human_selected")
        parser.add_argument('--assess_completeness', help='Compare to the concept completeness algorithm?', action='store_true')
        parser.add_argument('--num_iterations', help='Number of iterations for iterative algorithms',type=int, default=0)
        parser.add_argument('--selections_per_round', help='Concepts to select per round',type=int, default=0)
        parser.add_argument('--initial_concepts', help='Number of starting/initial concepts',type=int, default=0)
        parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

        args = parser.parse_args()

        seed = args.seed
        environment_string = args.environment_string
        training_timesteps = args.training_timesteps 
        gold_timesteps = args.gold_timesteps
        num_concepts_selected = args.num_concepts_selected
        selection_function = args.selection_function
        cbm_accuracy_by_concept = args.cbm_accuracy_by_concept
        cbm_std_by_concept = args.cbm_std_by_concept
        run_basic = args.run_basic
        run_iterative = args.run_iterative
        run_two_stage = args.run_two_stage
        run_imperfect = args.run_imperfect
        run_intervention = args.run_intervention
        intervention_probability = args.intervention_probability
        intervention_accuracy_by_concept = args.intervention_accuracy_by_concept
        target_abstraction = args.target_abstraction
        reward_error = args.reward_error
        concept_source = args.concept_source
        assess_completeness = args.assess_completeness
        num_iterations = args.num_iterations 
        selections_per_round = args.selections_per_round
        initial_concepts = args.initial_concepts
        out_folder = args.out_folder

    save_name = secrets.token_hex(4)  

In [9]:
if is_main:
        results = {}
        results['parameters'] = {'seed'      : seed,
                'environment_string'    : environment_string, 
                'training_timesteps': training_timesteps, 
                'gold_timesteps': gold_timesteps,
                'selection_function': selection_function,
                'num_concepts_selected': num_concepts_selected,
                'cbm_accuracy_by_concept': cbm_accuracy_by_concept,
                'cbm_std_by_concept': cbm_std_by_concept,
                'intervention_probability': intervention_probability,
                'intervention_accuracy_by_concept': intervention_accuracy_by_concept,
                'target_abstraction': target_abstraction,
                'reward_error': reward_error, 
                'concept_source': concept_source,
                'assess_completeness': assess_completeness,
                'num_iterations': num_iterations,
                'selections_per_round': selections_per_round, 
                'initial_concepts': initial_concepts,
                'run_basic': run_basic,
                'run_iterative': run_iterative, 
                'run_two_stage': run_two_stage, 
                'run_intervention': run_intervention,
                'run_imperfect': run_imperfect, 
        }
        print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'environment_string': 'mini_grid', 'training_timesteps': 1000000, 'gold_timesteps': 1000000, 'selection_function': 'q_value', 'num_concepts_selected': 24, 'cbm_accuracy_by_concept': None, 'cbm_std_by_concept': None, 'intervention_probability': 0, 'intervention_accuracy_by_concept': None, 'target_abstraction': 0.05, 'reward_error': 0, 'concept_source': 'human_selected_binary', 'assess_completeness': False, 'num_iterations': 0, 'selections_per_round': 0, 'initial_concepts': 0, 'run_basic': False, 'run_iterative': False, 'run_two_stage': False, 'run_intervention': True, 'run_imperfect': True}


In [10]:
if is_main:
    np.random.seed(seed)
    random.seed(seed)

### Basic Setup

In [11]:
if is_main:
    concept_list = get_concepts(environment_string,concept_source,seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env, additional_info = get_environment(environment_string, None, seed)   

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [10]:
if is_main:
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    
    if os.path.exists(model_name):
        print("Model exists!")
        groundtruth_model = PPO.load(model_name)
        additional_info['subset_concepts'] = get_concepts(environment_string,"human_selected",seed)
    else:
        if "cyclic" in environment_string or "tree" in environment_string or "glucose" in environment_string:
            policy = "MlpPolicy"
        else:
            policy = "CnnPolicy"
        
        if environment_string == "mimic":
            additional_info['subset_concepts'] = get_concepts(environment_string,"human_selected",seed)
            groundtruth_model = train_ppo_model(ground_truth_env,"mimic_raw",total_timesteps=gold_timesteps,policy=policy,)
        else:
            groundtruth_model = train_ppo_model(ground_truth_env,environment_string,total_timesteps=gold_timesteps,policy=policy)
        groundtruth_model.save(model_name)
    groundtruth_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,groundtruth_model,seed)
    results['ground_truth'] = {'reward':groundtruth_reward}
    print("Basic:",results['ground_truth']['reward'])

Model exists!


ALSA lib confmisc.c:767:(parse_card) cannot find card '0'
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_card_driver returned error: No such file or directory
ALSA lib confmisc.c:392:(snd_func_concat) error evaluating strings
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1246:(snd_func_refer) error evaluating name
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5220:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2642:(snd_pcm_open_noupdate) Unknown PCM default
ALSA lib confmisc.c:767:(parse_card) cannot find card '0'
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_card_driver returned error: No such file or directory
ALSA lib confmisc.c:392:(snd_func_concat) error evaluating strings
ALSA lib conf.c:4732:(_snd_config_evaluate) function snd_func_concat returned error: N

ValueError: Error: Unexpected observation shape (8, 2, 84, 84) for Box environment, please use (4, 84, 84) or (n_env, 4, 84, 84) for the observation shape.

### Basic Comparison

In [ ]:
if is_main:    
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,selection_function,concept_source)
    results['basic_comparison'] = {}
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))
    else:
        if selection_function == "q_value":
            if environment_string == "glucose":
                q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list,total_timesteps=20_000)
            else:
                q_estimates = rollout_q_estimates_td(groundtruth_model,ground_truth_gym_env,concept_list)
        elif selection_function == "policy":
            q_estimates = rollout_pi_estimates(groundtruth_model,ground_truth_gym_env,concept_list)
        pickle.dump(q_estimates,open(model_name,"wb"))

In [16]:
if is_main and run_basic:
    # Train a random policy
    if environment_string == "mimic":
        model = RandomAgent(GymnasiumWrapper(DummyVecEnv([lambda: ground_truth_gym_env])))
    else:
        model = RandomAgent(ground_truth_gym_env)
    random_reward = evaluate_model(environment_string,ground_truth_gym_env,additional_info,model,seed)
    results['basic_comparison']['random'] = {'reward':random_reward}
    print("Random:",results['basic_comparison']['random']['reward'])

In [20]:
if is_main and run_basic:
    # Train a random selector
    subset_concept, random_idx = random_selection(concept_list,num_concepts_selected)
    subset_concept = [concept_list[i] for i in random_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)    
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_random".format(environment_string))
    random_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['random_selection'] = {'reward':random_selection_reward, 'concepts': random_idx}
    print("Random Selection:",results['basic_comparison']['random_selection']['reward'])

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▃▁▁▁▁▁▁▂▂▂▂▃▃▁▁▃▂▂▂▁▄▁▁█▃▂▂▁▁▃▂▂▃▂▂▂▁▂▂▂
avg_norm_reward,▁▁▁▁▁▁▁▁▁▁▂▁▃▁▄▅▅▅██▇▅█▆▄▆▃█▇▇██████████
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▄▁▁▂▁▁▁▁▁█▃▁▁▁▄▃▁▃▂▁▂▂▂▂▂▃▁▄▄
ema_norm_reward,▁▁▁▁▁▁▁▁▁▃▄▅▆▅▅▅▅▆▇█▅▄▆▄▆▅▅▆▇▇▇█████████
entropy_loss,▁▂▂▂▃▃▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▅▅▄▇▇▇▇██
episode_length,▁▁▁▁▃▂▄▅▃▅▃▇▆▆█▆▆▆▄▇█▅▄▄▄▇▅▅▅▄███▆██████
episode_reward,▁▁▁▁▁▂▃▅▂█▆▆▃██▄▅▇▄▄█▄▃▅▅▅██████████████
explained_variance,████████████████████████████▆▆▆▅▆▄▂▃▅▅▄▁
value_loss,▇▆▆▅▅▄▃▄▄▄▂▂▅▇▇▆▆█▇▇▆▆▄▃▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁
approx_kl,0.0023
avg_norm_reward,500


Random Selection: 494.8453608247423


In [18]:
if is_main and run_basic:
    # Train a greedy selector
    subset_concept, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_greedy".format(environment_string))
    greedy_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['greedy'] = {'concepts': greedy_idx, 'reward':greedy_selection_reward}
    print("Greedy:",results['basic_comparison']['greedy']['reward'])


In [19]:
if is_main and run_basic:
    subset_concept, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_lp".format(environment_string))
    lp_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['lp'] = {'concepts': lp_idx, 'reward': lp_selection_reward}
    print("LP Selection:",results['basic_comparison']['lp']['reward'])

In [20]:
if is_main and run_basic:
    subset_concept, multiple_idx = multiple_lp_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_multiple".format(environment_string))
    multiple_selection_reward = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    results['basic_comparison']['multiple'] = {'concepts': multiple_idx, 'reward': multiple_selection_reward}
    print("Multiple Selection:",results['basic_comparison']['multiple']['reward'])

### Imperfect Concept Predictors

In [22]:
if is_main and run_imperfect:
    results['inaccurate_comparison'] = {}

In [25]:
if is_main and run_imperfect:
    greedy_inaccurate_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for (func,acc) in zip(concept_list,cbm_accuracy_by_concept)]
    subset_concept, greedy_idx = greedy_selection(modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_error_greedy".format(environment_string))
    greedy_inaccurate_reward = {
        'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
        'concepts': greedy_idx
    }

    if greedy_inaccurate_reward != {}:
        results['inaccurate_comparison']['greedy'] = greedy_inaccurate_reward
        print(greedy_inaccurate_reward)

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▅▃▁▂▁▁▁▁▁▁▂▆▅▇▁▂▂▁▃▁▃▅▂█▁▁▂▆▁▃▃▃▃▂▂▂▄▂▂▁
avg_norm_reward,▁▁▁▁▁▂▂▂▅▄▅▄▃▄▄▄▃▅▄▄▆▄▄█▅▅▄▅▇▄▆█████████
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▂▂▆▁▂▁▁▂▁▁▃▁▁▁▁▂█▁▂▁▁▂▂▅▂▃▃▂▁
ema_norm_reward,▁▁▁▁▁▁▂▂▃▃▅▅▅▅▄▇▇▄▆▄▄▄▇▆▇▇▄▆████████████
entropy_loss,▁▁▁▂▃▃▃▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▄▄▄▅▅▇▇▇██████████
episode_length,▁▁▂▁▁▂▁▅▇▇▃▆▅▆█▄▇▆▂▂█▅▇▃▇█▄▅▄▆██████████
episode_reward,▁▁▁▁▁▁▃▁▄▃▆▄▇▅█▅▇█▄▅▆▇▄▆▄▄▅█▅▅▅█▄███████
explained_variance,███████████████████████████▆▆█▃▄▃▅▂▃▁▁▃▂
value_loss,▄▄▇▇▅▅▅▄▄▄▆▆▃▄▄▃▅▇▆▆▃▅██▇▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁
approx_kl,0.00053
avg_norm_reward,500


{'binary': {'reward': 494.8453608247423, 'concepts': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]}}


In [23]:
if is_main and run_imperfect:
    lp_inaccurate_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
    _, lp_idx = lp_based_selection(ground_truth_gym_env,modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
    subset_concept = [modified_concept_predictors[i] for i in lp_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_error_lp".format(environment_string))
    lp_inaccurate_reward = { 'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),'concepts': lp_idx}

    if lp_inaccurate_reward != {}:
        results['inaccurate_comparison']['lp'] = lp_inaccurate_reward
        print(lp_inaccurate_reward)

In [24]:
if is_main and run_imperfect:
    multiple_lp_selection_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary(func,acc,seed) for func,acc in zip(concept_list,cbm_accuracy_by_concept)]
    subset_concept, multiple_idx = multiple_lp_selection(ground_truth_gym_env,modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_error_multiple".format(environment_string))
    multiple_lp_selection_reward = {}
    multiple_lp_selection_reward['reward'] = evaluate_model(environment_string,eval_env,additional_info,model,seed)
    multiple_lp_selection_reward['concepts'] = multiple_idx

    if multiple_lp_selection_reward != {}:
        results['inaccurate_comparison']['multiple_lp'] = multiple_lp_selection_reward
        print(multiple_lp_selection_reward)

### Intervention

In [26]:
if is_main and run_intervention and intervention_accuracy_by_concept is not None:
    results['intervention_comparison'] = {}

In [ ]:
if is_main and run_intervention:
    greedy_intervention_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
    _, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    subset_concept = [modified_concept_predictors[i] for i in greedy_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_intervention_greedy".format(environment_string))

    greedy_intervention_reward] = {
        'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
        'concepts': greedy_idx
    }

    if greedy_intervention_reward != {}:
        results['intervention_comparison']['greedy'] = greedy_intervention_reward
        print(greedy_intervention_reward)

In [ ]:
if is_main and run_intervention:
    lp_intervention_reward = {}
    modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
    _, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    subset_concept = [modified_concept_predictors[i] for i in lp_idx]
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_intervention_lp".format(environment_string))

    lp_intervention_reward = {
        'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
        'concepts': lp_idx
    }

    if lp_intervention_reward != {}:
        results['intervention_comparison']['lp'] = lp_intervention_reward
        print(lp_intervention_reward)

In [ ]:
if is_main and run_intervention:
    modified_concept_predictors = [inaccurate_concepts_binary_intervention(func,acc,intervene_acc,intervention_probability,seed+idx) for (func,acc,intervene_acc,idx) in zip(concept_list,cbm_accuracy_by_concept,intervention_accuracy_by_concept,list(range(len(concept_list))))]
    subset_concept, multiple_idx = multiple_lp_selection(ground_truth_gym_env,modified_concept_predictors,num_concepts_selected,selection_function,q_estimates,concept_source)
    env, eval_env, additional_info = get_environment(environment_string,subset_concept,seed)
    model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_intervention_multiple".format(environment_string))
    multiple_lp_intervention_reward = {
        'reward': evaluate_model(environment_string,eval_env,additional_info,model,seed),
        'concepts': multiple_idx
    }
    if multiple_lp_intervention_reward != {}:
        results['intervention_comparison']['multiple_lp'] = multiple_lp_intervention_reward
        print(multiple_lp_intervention_reward)

### Two-Stage Training

In [89]:
run_two_stage = True

In [139]:
if is_main and run_two_stage:
    env, eval_env, additional_info = get_environment(environment_string,concept_list,seed)
    gold_path = "../../results/models/env={}_training={}_seed={}_two_step.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(gold_path):
        gold_model = PPO.load(gold_path)
    else:
        gold_model = train_ppo_model(env,environment_string,total_timesteps=training_timesteps,policy="MlpPolicy",custom_name="{}_gold_two_stage".format(environment_string))
        gold_model.save(gold_path)
    

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▆▃▅▅▇▄▄▅▅▅▂▅▁▄▄▃▃▃▃▄▅▄▃▆▃▂▅▅▅▂▃▄▅▃▃█▂▂▆▄
avg_norm_reward,▂▁▂▂▃▆▇█▅▆▄▆▇▆█▅▄▂▄▆▄▄▇▅▂▅▅█▆▇▅█▅▆▅▅▇█▆▆
clip_fraction,█▃▃▂▂▄██▅▅▃▃▄▃▃▃▃▃▂▂▄▄▃▂▅▂▂▁▂▆▂▂▂█▃▅▃▄▄▁
ema_norm_reward,▁▁▁▂▃▅▆▆▇▅▆▇▆▅▄▄▄▄▅▆▆▆▆▆▄▇▅▅▆▅█▇▅▇▇▇▇▅▅▇
entropy_loss,▁▃▄▆▆▆▇▇▆▇▇▇▇███▇▆▆▆▇▇▇▇█▆▆▆▆▆▆▆▇▆▆▆▇▆▇▇
episode_length,▁▂▁▁▁▁▂▁▂▅▄▅▄█▅▄▄▅▃▅▃▅▅▃▅▆▆▅▅▃▆▄▄▆█▇▄█▆▅
episode_reward,▁▁▁▁▁▄▅▄▃█▄▃▄▄▄▃▄▅▇▆▄█▇█▄▅▄█▃▅▆▇▆▆▄▆▇▆▅█
explained_variance,▅▆▄▃▃▇▃▃▃▃▃▃▃▃▃▃▃▃▃▃▁▄▇▄▅▄▇▅█▅███▆▇▅▅▅▅▇
value_loss,▃▃▃▃▃▃▄▃▂▄▅▄▆▅▄▅▄▅▆▄█▄▄▄▄█▄▂▄▁▄▃▃▂▅▄▄▄▂▃
approx_kl,0.00372
avg_norm_reward,500


In [124]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    concept_predictor, acc_list = train_concept_predictor(ground_truth_gym_env,gold_model,concept_list,list(range(len(concept_list))),epochs=100)
    results['two_stage'] = {}
    results['two_stage']['accuracy'] = acc_list.tolist()

Episode 10/25, steps: 500, samples collected: 6133
  Estimated memory: 82.5 MB
Episode 20/25, steps: 500, samples collected: 12115
  Estimated memory: 163.0 MB

✅ Collection complete!
X shape: (15059, 2, 84, 84), dtype: uint8, size: 202.7 MB
Y shape: (15059, 4), dtype: float32
Y contains 4 features per sample
X_data shape: (15059, 2, 84, 84), dtype: uint8
Y_data shape: (15059, 12), dtype: float32
📉 Epoch 1/100 | Train Loss 0.3986 | Val Loss 0.3587 | Val F1 0.8298 | Took 1.76s
📉 Epoch 2/100 | Train Loss 0.3345 | Val Loss 0.3399 | Val F1 0.8388 | Took 1.75s
📉 Epoch 3/100 | Train Loss 0.3173 | Val Loss 0.3309 | Val F1 0.8439 | Took 1.85s
📉 Epoch 4/100 | Train Loss 0.3047 | Val Loss 0.3336 | Val F1 0.8394 | Took 1.77s
📉 Epoch 5/100 | Train Loss 0.2960 | Val Loss 0.3276 | Val F1 0.8462 | Took 1.69s
📉 Epoch 6/100 | Train Loss 0.2871 | Val Loss 0.3341 | Val F1 0.8418 | Took 1.76s
📉 Epoch 7/100 | Train Loss 0.2804 | Val Loss 0.3290 | Val F1 0.8506 | Took 1.82s
📉 Epoch 8/100 | Train Loss 0.2743

In [16]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    greedy_two_stage = {}
    greedy_concepts, greedy_idx = greedy_selection(concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, greedy_concepts, seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=greedy_idx)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_two_stage_greedy".format(environment_string))    
    greedy_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['greedy'] = {'reward': greedy_two_stage_reward, 'concepts': greedy_idx}
    print(greedy_two_stage_reward)



16 concepts


/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/environment_wrappers.py:161: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:275.)
  obs_array = torch.Tensor([processed_obs]).cuda()
/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/environment_wrappers.py:161: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:275.)
  obs_array = torch.Tensor([processed_obs]).cuda()
/usr0/home/naveenr/projects/concept_decisions/concept_abstraction/environment_wrappers.py:161: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider c

approx_kl,▇▄▄▄▄▃▃▃▃▅▇▅▃▂▂▃▆▅▅▁▅▄▃▃▃▄▂▃▅▅▂▅▅▅▅▂▁▃█▃
avg_norm_reward,▁▃▃▃▄▄▂▄▃▃▃▆▂▃▄▄▃█▄▃▆▃▆▄▄▅▅▃▂▃▃▃▃▄▄▅▃▅▅▆
clip_fraction,▁▁▁▁▂▄▃▄▄▄▄▃▁▂▃▃▁█▃▁▁▃▂▁▄▂▂▁▂▂▄▄▂▂▂▁▂▅▂▃
ema_norm_reward,▁▄▅▄▄▅▅▄▆▅▆▅▄▆▆▆▅▄█▆▅▅▄▇█▆▆▆▅▆▆▆▆█▅▆▃▆▅▅
entropy_loss,▂▁▅▅▅▅▅▆▆▆▇▇▇██▆▆▅▆▆▅▅▆▆▆▅▅▄▅▆▆▆▆▇▆▇▇▇▆▆
episode_length,▁▁▃▄▃▅▂▂▄▂▃▂▂▄▅▄▅▃▃▂▃▂█▂▃▃▄▆▂▂▃▃▃▂▄▂▃▃▃▃
episode_reward,▁▃▃▂▄▅▃█▄▂▃▁▃▄▆▃▃▅▃▃▄▄█▂▃▄▆▄▃▂▃▄▃▂▆▄▃▆▂▂
explained_variance,▅▁▂▄▅▇▆▆▇▇▇▇▇▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇▇██▇▇▇█
value_loss,▁▂▃▄▅▇▇▇▇▇▆▅▆▆▆▆▆▇▆███▇▇▆▆▇██▇▇▇▅▇▇▇▇█▇▇
approx_kl,0.00356
avg_norm_reward,55


64.88657105606258


In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    lp_concepts, lp_idx = lp_based_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, lp_concepts, seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=lp_idx)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_two_stage_lp".format(environment_string))    
    lp_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['lp'] = {'reward': lp_two_stage_reward, 'concepts': lp_idx}
    print(lp_two_stage_reward)



In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    multiple_concepts, multiple_idx = multiple_lp_selection(ground_truth_gym_env,concept_list,num_concepts_selected,selection_function,q_estimates,concept_source)

    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, multiple_concepts, seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=multiple_idx)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_two_stage_multiple".format(environment_string))    
    multiple_iterative_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['multiple'] = {'reward': multiple_iterative_two_stage_reward, 'concepts': multiple_idx}
    print(multiple_iterative_two_stage_reward)



In [ ]:
if is_main and not isinstance(ground_truth_env, ConceptEnv) and run_two_stage:
    acc_list = results['two_stage']['accuracy']
    top_k_idx = np.argsort(acc_list)[-num_concepts_selected:].tolist()
    top_k_concepts = [concept_list[i] for i in top_k_idx]
    
    two_stage_env, two_stage_gym_env, additional_info = get_environment(environment_string, top_k_concepts, seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=top_k_idx)   
    model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_two_stage_topk".format(environment_string))    
    top_k_two_stage_reward = evaluate_model(environment_string,two_stage_gym_env,additional_info,model,seed)

    results['two_stage']['top_k'] = {'reward': top_k_two_stage_reward, 'concepts': top_k_idx}
    print(top_k_two_stage_reward)



## Save Data

In [ ]:
if is_main:
    save_path = get_save_path(out_folder,save_name)

In [ ]:
if is_main:
    delete_duplicate_results(out_folder,"",results)

In [ ]:
if is_main:
    json.dump(results,open('../../results/'+save_path,'w'))

In [ ]:
if is_main:
    ground_truth_env.close()
    ground_truth_gym_env.close()
    env.close()
    eval_env.close()